In [2]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
print("Step 1: Loading raw dataset...")
df = pd.read_csv('Coffee Shop Sales.csv')
print(f"Done. Loaded {len(df)} raw transaction rows.")

Step 1: Loading raw dataset...
Done. Loaded 149116 raw transaction rows.


In [5]:
# 2. Extract Date, Time, and Revenue Features
print("\nStep 2: Preprocessing dates and calculating transaction revenue...")
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.day_name()
df['is_weekend'] = df['transaction_date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
df['hour'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour

# Calculate individual transaction revenue
df['revenue'] = df['transaction_qty'] * df['unit_price']
print("Done. Calculated revenue per transaction and extracted time features.")


Step 2: Preprocessing dates and calculating transaction revenue...
Done. Calculated revenue per transaction and extracted time features.


In [7]:
# 3. Create Chronological Shift Mapping
print("\nStep 3: Assigning shifts for chronological tracking...")
def get_shift_info(hour):
    if 6 <= hour < 12: 
        return 'Morning', 1
    elif 12 <= hour < 17: 
        return 'Afternoon', 2
    else: 
        return 'Evening', 3

df[['shift', 'shift_num']] = df['hour'].apply(lambda x: pd.Series(get_shift_info(x)))
print("Done. Shift numbers assigned (1-Morning, 2-Afternoon, 3-Evening).")


Step 3: Assigning shifts for chronological tracking...
Done. Shift numbers assigned (1-Morning, 2-Afternoon, 3-Evening).


In [9]:
# 4. Aggregation (The "Revenue" Level)
print("\nStep 4: Aggregating into shift-level financial data...")
# We sum both quantity (cups) and revenue
revenue_agg = df.groupby([
    'transaction_date', 
    'store_location', 
    'month', 
    'day_of_week', 
    'is_weekend', 
    'shift', 
    'shift_num'
]).agg({
    'transaction_qty': 'sum',
    'revenue': 'sum'
}).reset_index()

# Rename for clarity
revenue_agg = revenue_agg.rename(columns={
    'transaction_qty': 'total_cups',
    'revenue': 'total_revenue'
})
print(f"Done. Data aggregated into {len(revenue_agg)} shift financial records.")


Step 4: Aggregating into shift-level financial data...
Done. Data aggregated into 1623 shift financial records.


In [11]:
# 5. Create the Lag Features (Financial Memory)
print("\nStep 5: Sorting and creating chronological 'prev_revenue' and 'prev_cups'...")
# Sort by Location -> Date -> Shift Number to ensure correct chronological sequence
revenue_agg = revenue_agg.sort_values(['store_location', 'transaction_date', 'shift_num'])

# Calculate revenue and cups from the previous shift
revenue_agg['prev_revenue'] = revenue_agg.groupby('store_location')['total_revenue'].shift(1).fillna(0)
revenue_agg['prev_cups'] = revenue_agg.groupby('store_location')['total_cups'].shift(1).fillna(0)
print("Done. Lag features generated to capture historical momentum.")


Step 5: Sorting and creating chronological 'prev_revenue' and 'prev_cups'...
Done. Lag features generated to capture historical momentum.


In [12]:
# 6. Export the cleaned dataset
print("\nStep 6: Exporting cleaned CSV for Model 2 Training...")
revenue_agg.to_csv('Model2_Revenue_Data.csv', index=False)
print("Done. File saved as 'Model2_Revenue_Data.csv'.")

print("\n" + "="*40)
print("FEATURE ENGINEERING FOR MODEL 2 COMPLETE")
print("="*40)
print(revenue_agg[['transaction_date', 'shift', 'total_cups', 'total_revenue', 'prev_revenue']].head(10))


Step 6: Exporting cleaned CSV for Model 2 Training...
Done. File saved as 'Model2_Revenue_Data.csv'.

FEATURE ENGINEERING FOR MODEL 2 COMPLETE
   transaction_date      shift  total_cups  total_revenue  prev_revenue
2        2023-01-01    Morning          31          98.90          0.00
0        2023-01-01  Afternoon         156         466.35         98.90
1        2023-01-01    Evening          90         303.15        466.35
11       2023-01-02    Morning          38         112.30        303.15
9        2023-01-02  Afternoon         163         503.85        112.30
10       2023-01-02    Evening         100         309.35        503.85
20       2023-01-03    Morning          35         109.70        309.35
18       2023-01-03  Afternoon         166         522.30        109.70
19       2023-01-03    Evening          86         270.75        522.30
29       2023-01-04    Morning          23          72.40        270.75
